In [1]:
from langchain_core.documents import Document

In [ ]:
doc=Document(page_content="This is the content for RAG",
 metadata={
    "source": "example.com",
    "author": "John Doe",
    "date": "2021-01-01"
    })

print(doc.page_content)
print(doc.metadata)

In [ ]:
### Text Loader

from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/textfiles/ishan.txt")
docs = loader.load()
print(docs)




In [ ]:
### PDF Loader
from langchain_community.document_loaders import PyMuPDFLoader
loader=PyMuPDFLoader("../data/pdf/fire_alarm.pdf")  
docs=loader.load()
print(docs)

In [ ]:
##Directory Loader for TExt Files
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader("../data/textfiles",
glob="*.txt",
loader_cls=TextLoader,
loader_kwargs={"encoding":"utf-8"}
)
docs=dir_loader.load()
print(docs)

In [ ]:
##Directory Loader for TExt Files
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader("../data/pdf",
glob="*.pdf",
loader_cls=PyMuPDFLoader,
)
docs=dir_loader.load()
print(docs)

In [ ]:
### PDF Chunking helpers
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def load_pdf(path: str) -> list[Document]:
    """Load a single PDF into one Document per page."""
    return PyMuPDFLoader(path).load()


def load_pdf_dir(dir_path: str, glob: str = "*.pdf") -> list[Document]:
    """Load every PDF in a directory."""
    return DirectoryLoader(dir_path, glob=glob, loader_cls=PyMuPDFLoader).load()


def chunk_documents(
    docs: list[Document],
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Split loaded documents into overlapping chunks for RAG.

    Metadata from the source page is preserved on every chunk, and a
    `chunk_id` is added so chunks stay traceable back to their document.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
    return chunks


def chunk_pdf(
    path: str,
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Load a single PDF and return its chunks."""
    return chunk_documents(load_pdf(path), chunk_size, chunk_overlap)


def chunk_pdf_dir(
    dir_path: str,
    chunk_size: int = 800,
    chunk_overlap: int = 100,
) -> list[Document]:
    """Load all PDFs in a directory and return their chunks."""
    return chunk_documents(load_pdf_dir(dir_path), chunk_size, chunk_overlap)

In [ ]:
### Try the chunker on the fire alarm PDF
chunks = chunk_pdf("../data/pdf/lesson.pdf", chunk_size=500, chunk_overlap=80)

print(f"Total chunks: {len(chunks)}")
for c in chunks[:3]:
    print("-" * 60)
    print("page:", c.metadata.get("page"), "| chunk_id:", c.metadata.get("chunk_id"))
    print("chars:", len(c.page_content))
    print(c.page_content[:200], "...")

In [4]:
### Open-source embeddings + ChromaDB storage
from pathlib import Path

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DEFAULT_CHROMA_DIR = "../data/chroma_db"
DEFAULT_COLLECTION = "portfolio"


def get_embeddings(model_name: str = DEFAULT_EMBEDDING_MODEL) -> HuggingFaceEmbeddings:
    """Create a local open-source embedding model (runs on CPU, no API key)."""
    return HuggingFaceEmbeddings(model_name=model_name)


def create_vector_store(
    chunks: list[Document],
    persist_directory: str = DEFAULT_CHROMA_DIR,
    collection_name: str = DEFAULT_COLLECTION,
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> Chroma:
    """Embed chunks and store them in a persistent ChromaDB collection."""
    Path(persist_directory).mkdir(parents=True, exist_ok=True)
    embeddings = get_embeddings(model_name)

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )


def load_vector_store(
    persist_directory: str = DEFAULT_CHROMA_DIR,
    collection_name: str = DEFAULT_COLLECTION,
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> Chroma:
    """Load an existing ChromaDB collection from disk."""
    return Chroma(
        persist_directory=persist_directory,
        collection_name=collection_name,
        embedding_function=get_embeddings(model_name),
    )


def search_vector_store(
    vector_store: Chroma,
    query: str,
    k: int = 3,
) -> list[Document]:
    """Return the top-k most similar chunks for a query."""
    return vector_store.similarity_search(query, k=k)

In [13]:
### Embed PDF chunks and store in ChromaDB
pdf_chunks = chunk_pdf("../data/pdf/lesson.pdf", chunk_size=500, chunk_overlap=80)

vector_store = create_vector_store(
    chunks=pdf_chunks,
    persist_directory="../data/chroma_db",
    collection_name="portfolio",
)

print(f"Stored {vector_store._collection.count()} chunks in ChromaDB")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6263.33it/s]


Stored 236 chunks in ChromaDB


In [19]:
### Test retrieval from ChromaDB
query = " What is the Assignment?"
results = search_vector_store(vector_store, query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("page:", doc.metadata.get("page"), "| chunk_id:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:500], "...\n")

Query:  What is the Assignment?

Result 1
page: 139 | chunk_id: 222
• Define the term semaphore. How does semaphore help in dining philosophers
problem? Explain.
• Explain the Peterson’s concept for the solution of critical section problem.
• What is critical section problem? Why executing critical selection must be mutual
exclusive? Explain.
• What is lock variable? Discuss its working and problems associated with it in detail.
• Show how sleep and wake up solution is better than busy waiting solution for the critical
section problem. ...

Result 2
page: 137 | chunk_id: 217
Assignment 2
• How does process differ from program? Explain process state with the help of 
block diagram.
• Why some process requires high priority? What would happen if all processes have some 
the priority? Mention merits and demerits of assigning priority on process.
• Do you think a process can exist without any state? Justify your view with the help of 
process state transition diagram.
• For each of the fol